In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import logging
import os
import sys
sys.path.insert(0, "/".join(["..", "python"]))

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import rasterio
from rasterio.io import MemoryFile
from rasterio.transform import Affine, from_origin

from gridr.chain.fft_filtering_chain import (
    check_oa_strip_size,
    extended_extent,
    fft_filtering_oa_strip_chain,
)
from gridr.core.convolution.fft_filtering import (
    fft_array_filter,
    fft_array_filter_output_shape,
    pad_kernel_to_odd,
)
from gridr.core.utils import chunks
from gridr.misc.mandrill import mandrill
from notebook_utils import plot_im, mpl_plot_wrapper
IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
import notebook_utils
notebook_utils.FORCE_MPL_INTERACTIVE = True
if not IN_DOC_BUILD and not notebook_utils.FORCE_MPL_INTERACTIVE:
    from bokeh.io import output_notebook  # enables plot interface in J notebook
    output_notebook()


def gaussian_kernel_2d(shape, sigma):
    """
    Generates a normalized 2D Gaussian kernel.

    :param shape: Kernel size (int for a square kernel, or tuple (height, width))
    :param sigma: Standard deviation of the Gaussian (float or tuple (sigma_y, sigma_x))
    :return: a tuple containing:
        - 1D Numpy array representing x coordinates
        - 1D Numpy array representing y coordinates
        - 2D NumPy array representing the kernel
    """
    # Handle scalar or tuple inputs for shape and sigma
    if isinstance(shape, int):
        h = w = shape
    else:
        h, w = shape

    if isinstance(sigma, (int, float)):
        sigma_y = sigma_x = sigma
    else:
        sigma_y, sigma_x = sigma

    # Create centered coordinates (e.g., size 5 -> [-2, -1, 0, 1, 2])
    y = np.arange(-(h // 2), h // 2 + 1 if h % 2 != 0 else h // 2)
    x = np.arange(-(w // 2), w // 2 + 1 if w % 2 != 0 else w // 2)

    # 2D Grid
    x_grid, y_grid = np.meshgrid(x, y)

    # Calculate 2D Gaussian function
    kernel = np.exp(-0.5 * ((x_grid / sigma_x) ** 2 + (y_grid / sigma_y) ** 2))

    # Normalize so the sum of all elements equals 1
    kernel_sum = np.sum(kernel)
    if kernel_sum != 0:
        kernel /= kernel_sum

    return x, y, kernel


def strip_layout(win, kernel, strip_size, raster_height):
    """Strip decomposition of a production window, as the chain computes it.

    The strip size is passed through `check_oa_strip_size` and the cuts through
    `chunks.get_chunk_boundaries`, so the layout follows the implementation. The
    read extents come from `extended_extent`. The written blocks are derived from
    the kernel margin: each internal cut is covered by a seam of ``k + 1`` rows
    taken from the overlap-add buffer, the rest by the middle of a strip.

    :param win: production window as (2, 2) inclusive (first, last) bounds
    :param kernel: the filter, padded to an odd size here as the chain does
    :param strip_size: the requested strip height, in window rows
    :param raster_height: height of the input raster, used to clip the reads
    :return: a tuple (list of per strip dictionaries, effective strip size,
        kernel margin in rows)
    """
    first_row, last_row = int(win[0][0]), int(win[0][1])
    nrow = last_row - first_row + 1
    kernel = pad_kernel_to_odd(np.asarray(kernel), axes=None)
    krow = kernel.shape[0]
    margin = krow // 2

    effective = check_oa_strip_size(nrow=nrow, kernel=kernel, strip_size=strip_size)
    if effective == 0:
        bounds = [(0, nrow)]  # single call, no overlap-add
    else:
        bounds = chunks.get_chunk_boundaries(nsize=nrow, chunk_size=effective,
                                             merge_last=True)
    last_idx = len(bounds) - 1

    strips = []
    for idx, (lower, upper) in enumerate(bounds):
        read = extended_extent(first_row + lower, first_row + upper - 1,
                               raster_height, margin,
                               extend_before=idx == 0, extend_after=idx == last_idx)
        middle = (0 if idx == 0 else lower + margin + 1,
                  nrow - 1 if idx == last_idx else upper - margin - 2)
        seam = None if idx == 0 else (lower - margin - 1, lower + margin)
        strips.append(dict(idx=idx, cut=(lower, upper - 1), read=read,
                           middle=middle, seam=seam))

    # The written blocks tile the window: no row is written twice, none is skipped.
    written = sorted([s["seam"] for s in strips if s["seam"] is not None]
                     + [s["middle"] for s in strips])
    assert written[0][0] == 0 and written[-1][1] == nrow - 1
    assert all(nxt[0] == cur[1] + 1 for cur, nxt in zip(written, written[1:]))
    return strips, effective, margin


@mpl_plot_wrapper
def plot_strip_layout(win, kernel, strip_size, raster_height, dpi=100, prefix=None):
    """Show what each strip reads and what it writes, in input raster rows.

    The left lane holds the reads, each strip drawn solid over its own rows and
    hatched over the kernel margins it reads around the window. The right lane
    holds the written blocks: the middle of each strip, and the seams rebuilt
    from the overlap-add buffer at the internal cuts.

    :param win: production window as (2, 2) inclusive (first, last) bounds
    :param kernel: the filter given in the spatial domain
    :param strip_size: the requested strip height, in window rows
    :param raster_height: height of the input raster
    :param dpi: resolution of the figure
    :return: the matplotlib figure
    """
    strips, effective, margin = strip_layout(win, kernel, strip_size, raster_height)
    first_row, last_row = int(win[0][0]), int(win[0][1])
    colors = plt.get_cmap("tab10").colors

    fig, ax = plt.subplots(figsize=(7.2, 5.2), dpi=dpi)
    lane_read, lane_write, w = 0.0, 1.6, 1.0

    for s in strips:
        col = colors[s["idx"] % len(colors)]
        r_lo, r_hi = s["read"]
        c_lo, c_hi = s["cut"]
        top, bottom = first_row + c_lo, first_row + c_hi

        # Kernel margins, drawn only where the read goes past the strip
        if r_lo < top:
            ax.add_patch(Rectangle((lane_read, r_lo), w, top - r_lo,
                                   facecolor=col, alpha=0.18, edgecolor=col,
                                   linewidth=1.0, hatch="///"))
        if r_hi > bottom:
            ax.add_patch(Rectangle((lane_read, bottom + 1), w, r_hi - bottom,
                                   facecolor=col, alpha=0.18, edgecolor=col,
                                   linewidth=1.0, hatch="///"))
        ax.add_patch(Rectangle((lane_read, top), w, c_hi - c_lo + 1,
                               facecolor=col, alpha=0.45, edgecolor=col,
                               linewidth=1.4))
        ax.text(lane_read + w / 2, (top + bottom) / 2,
                f"strip {s['idx']}\n{c_hi - c_lo + 1} rows",
                ha="center", va="center", fontsize=8)

        # Written blocks
        m_lo, m_hi = s["middle"]
        ax.add_patch(Rectangle((lane_write, first_row + m_lo), w, m_hi - m_lo + 1,
                               facecolor=col, alpha=0.45, edgecolor=col,
                               linewidth=1.4))
        ax.text(lane_write + w / 2, first_row + (m_lo + m_hi) / 2,
                f"middle\n{m_hi - m_lo + 1} rows", ha="center", va="center",
                fontsize=7.5)
        if s["seam"] is not None:
            g_lo, g_hi = s["seam"]
            ax.add_patch(Rectangle((lane_write, first_row + g_lo), w,
                                   g_hi - g_lo + 1, facecolor="0.55", alpha=0.75,
                                   edgecolor="0.25", linewidth=1.4))
            ax.text(lane_write + w + 0.08, first_row + (g_lo + g_hi) / 2 + 0.5,
                    f"seam, {g_hi - g_lo + 1} rows", ha="left", va="center",
                    fontsize=7.5, color="0.25")
        ax.plot([lane_read, lane_read + w], [top] * 2, color="0.3",
                linewidth=0.8, linestyle=(0, (4, 3)))

    ax.add_patch(Rectangle((-0.15, first_row), lane_write + w + 0.3,
                           last_row - first_row + 1, fill=False,
                           edgecolor="tab:orange", linewidth=1.6, zorder=6))
    ax.annotate("production window", (lane_write + w + 0.15, first_row),
                fontsize=8, color="tab:orange", ha="right", va="bottom",
                xytext=(0, 4), textcoords="offset points")

    ax.set_xlim(-0.4, lane_write + w + 1.2)
    ax.set_ylim(last_row + margin + 6, first_row - margin - 6)
    ax.set_xticks([lane_read + w / 2, lane_write + w / 2])
    ax.set_xticklabels(["read from the raster", "written to the output"], fontsize=9)
    ax.set_ylabel("input raster row", fontsize=9)
    ax.set_yticks(sorted({first_row, last_row,
                          *(first_row + s["cut"][0] for s in strips)}))
    ax.tick_params(labelsize=8)
    ax.set_title(f"strip_size = {strip_size} -> {effective} used, "
                 f"{len(strips)} strips, kernel margin {margin} rows", fontsize=10)
    fig.text(0.5, 0.015, "hatched: kernel margins read around the window, "
             "absent at the internal cuts", ha="center", fontsize=8, color="0.35")
    fig.tight_layout(rect=[0, 0.05, 1, 1])
    return fig


mandrill_0 = mandrill[0]

# The `gaussian_kernel_2d` function implements the formula given in the first tutorial.
gaussian_sigma = 4.
x, y, kernel = gaussian_kernel_2d(shape=int(gaussian_sigma * 6 + 1), sigma=gaussian_sigma)

# Strips, memory and tracing

This last chain tutorial looks at `strip_size`, the argument that decides how the
production window is cut and how much memory a run needs. The `logger` argument comes with
it, since the debug trace is where the cut becomes observable.

**What you'll learn**

- How `strip_size` cuts the window, and what each strip reads and writes
- How to trace the loop with `logger`
- When the chain decides on its own to process the window in a single call
- What sizes the memory of a run
- Which restrictions come from the striping, and which come from the SciPy backends

## Setting things up

The input raster and the two helpers are the ones of the previous tutorial. The window
sits well inside the raster this time, so that the kernel margins are read from real
neighbours on all four sides.

In [ ]:
import logging
import sys

import numpy as np
from rasterio.io import MemoryFile
from rasterio.transform import Affine, from_origin

from gridr.chain.fft_filtering_chain import (
    check_oa_strip_size,
    fft_filtering_oa_strip_chain,
)
from gridr.core.convolution.fft_filtering import fft_array_filter_output_shape
from gridr.misc.mandrill import mandrill

mandrill_0 = mandrill[0]

profile_in = {
    "driver": "GTiff",
    "height": mandrill_0.shape[0],
    "width": mandrill_0.shape[1],
    "count": 1,
    "dtype": mandrill_0.dtype,
    "crs": "EPSG:32631",
    "transform": from_origin(400000.0, 5000000.0, 1.0, 1.0),
}

memfile_in = MemoryFile()
with memfile_in.open(**profile_in) as ds:
    ds.write(mandrill_0, 1)

ds_in = memfile_in.open()


def open_output(shape, transform, dtype="float32"):
    """Open an in-memory dataset sized and georeferenced for the filtering."""
    memfile = MemoryFile()
    profile = profile_in | {
        "height": shape[0],
        "width": shape[1],
        "dtype": dtype,
        "transform": transform,
    }
    return memfile, memfile.open(**profile)


def read_back(memfile, ds_out):
    """Flush the output dataset and read its single band back."""
    ds_out.close()
    with memfile.open() as src:
        return src.read(1)

In [ ]:
win = np.asarray([[40, 231], [160, 351]])
boundary = "reflect"
out_mode = "same"
strip_size = 55

out_shape = fft_array_filter_output_shape(
    (ds_in.height, ds_in.width),
    kernel,
    win=win,
    boundary=boundary,
    out_mode=out_mode,
)
row_first, col_first = int(win[0, 0]), int(win[1, 0])
transform_win = ds_in.transform * Affine.translation(col_first, row_first)

out_shape

## How the window is cut

`strip_size` is a target height, counted in rows of the production window. The window is
cut into strips of that height, and the last two are merged when the last one comes up
short, so no strip is shorter than the requested size.

Each strip is read with the kernel margins added at the edges of the window, and without
them at the internal cuts. A strip therefore sees no data belonging to its neighbours, and
the contributions missing on either side of a cut are exactly what the overlap-add buffer
carries over. Each strip then writes two disjoint blocks: the seam it shares with the
previous strip, rebuilt from the buffer, and its own middle.

In [ ]:
_ = plot_strip_layout(win, kernel, strip_size, ds_in.height,
                      prefix="chain_003_strip_layout")

The figure reads from left to right. On the left, what each strip reads: its own rows in
solid colour, and the kernel margins in hatched colour, present only above the first strip
and below the last one. On the right, what reaches the output: the middle of each strip in
the same colour, and the seams in grey.

A seam is `k + 1` rows tall, one more than the kernel, because the tail of one strip and
the head of the next overlap on `k - 1` rows and are stacked with a one row offset. The
middles and the seams tile the window exactly, which is what `strip_layout` asserts before
drawing anything.

## Tracing the loop

The chain takes a `logger` argument and emits the whole decomposition at the `DEBUG`
level: the window it received, the two output shapes it computed, then one line per strip
with the rows it covers, the array it read, the window expressed inside that array, the
boundary policy it applied, and the rows it wrote.

A logger is built below with a plain formatter and a handler on `sys.stdout`.

In [ ]:
logger = logging.getLogger("gridr.tutorial")
logger.setLevel(logging.DEBUG)
logger.propagate = False

handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(message)s"))
logger.handlers = [handler]

In [ ]:
memfile_out, ds_out = open_output(out_shape, transform_win)

fft_filtering_oa_strip_chain(
    ds_in=ds_in,
    ds_out=ds_out,
    band=1,
    kernel=kernel,
    boundary=boundary,
    out_mode=out_mode,
    win=win,
    strip_size=strip_size,
    round_out=False,
    logger=logger,
)

filtered = read_back(memfile_out, ds_out)

The trace matches the figure line by line. The boundary policy printed for the middle
strip is `"none"` on both sides of the row axis, while the first and last strips keep the requested `"reflect"` on their
outer side. The written row ranges alternate between an overlap area and a non-overlap
area, and they are the seams and the middles of the figure.

In [ ]:
plot_im({"window": mandrill_0[40:232, 160:352], "filtered": filtered},
        prefix="chain_003_filtered")

## When the chain falls back to a single call

The requested strip size is not always the one used. `check_oa_strip_size` returns `0`,
meaning a single call on the whole window, in three cases: when the window is shorter than
twice the strip size, since there would be little to gain; when the kernel covers half the
window or more, since the seams would eat the strips; and when the strip is shorter than
the kernel, since a strip has to carry its own support.

In [ ]:
for requested in (0, 16, 24, 55, 96, 128, 512):
    effective = check_oa_strip_size(nrow=192, kernel=kernel, strip_size=requested)
    print(f"strip_size={requested:4d}  ->  {effective:4d} used"
          f"{'   (single call)' if effective == 0 else ''}")

With the 192 row window used here, a request of 24 rows falls back because it is shorter
than the 25 row kernel, and a request of 128 falls back because the window is not twice
that. Between the two, a request of 55 or 96 is honoured.

The fallback path calls `fft_array_filter` once on the window extended by its margins, so
the result is the same as the striped one, up to the floating-point tolerance discussed in
the overview. Passing `strip_size=0` asks for that path explicitly.

## What sizes the memory

A striped run holds three arrays at once: the read buffer, sized for the tallest strip
plus its margins; the array the backend returns for the current strip, which is the
convolution in full mode; and the overlap-add buffer, `k + 1` rows tall and as wide as the
output. None of them depends on the height of the raster, which is the point of the chain,
and all of them grow with the width of the window.

The rough sizes below ignore the workspace SciPy allocates for the transforms themselves,
which is of the same order as the convolution array.

In [ ]:
def strip_memory(nrow, ncol, kernel, strip_size, dtype_in, dtype_work="float32"):
    """Rough size of the arrays a strip run holds at once, in MiB.

    :param nrow: height of the production window
    :param ncol: width of the production window
    :param kernel: the filter, assumed odd sized
    :param strip_size: the requested strip height
    :param dtype_in: dtype of the input raster
    :param dtype_work: working dtype of the convolution
    :return: a dictionary of sizes in MiB, and the strip size actually used
    """
    krow, kcol = kernel.shape
    mrow, mcol = krow // 2, kcol // 2
    effective = check_oa_strip_size(nrow=nrow, kernel=kernel, strip_size=strip_size)
    chunk = nrow if effective == 0 else effective
    read = (chunk + 2 * mrow) * (ncol + 2 * mcol) * np.dtype(dtype_in).itemsize
    conv = ((chunk + 2 * mrow + krow - 1) * (ncol + 2 * mcol + kcol - 1)
            * np.dtype(dtype_work).itemsize)
    oa = (krow + 1) * ncol * np.dtype(dtype_work).itemsize
    return {"read": read / 2 ** 20, "convolution": conv / 2 ** 20,
            "overlap-add": oa / 2 ** 20, "strip used": effective}


# A large window, where the strip size is the only thing bounding the memory.
for requested in (256, 1024, 4096, 0):
    sizes = strip_memory(nrow=40000, ncol=20000, kernel=kernel,
                         strip_size=requested, dtype_in=mandrill_0.dtype)
    total = sizes["read"] + sizes["convolution"] + sizes["overlap-add"]
    print(f"strip_size={requested:5d}  ->  used {sizes['strip used']:5d}, "
          f"about {total:8.1f} MiB held at once")

The last line asks for a single call on a 40000 by 20000 window, which is what a chain is
meant to avoid. Between the others, the memory follows the strip size almost linearly,
which makes it the knob to turn against the memory available on the machine.

Two other settings weigh on the same total. Pinning `dtype="float32"` halves the
convolution array and the overlap-add buffer against the `float64` NumPy would otherwise
promote to. Narrowing the window with `win` reduces every term, since the width appears in
all three.

## What currently striping restricts

Two restrictions of the chain were listed in the overview, and only one of them comes
from the striping.

`out_mode="valid"` raises `NotImplementedError` whatever the strip size, the chain knowing
only how to place the `"same"` and `"full"` regions in the output.

A `"wrap"` boundary on the row axis is refused only when the window actually ends up
striped. Periodic continuation asks for rows at the far end of the window, which a strip
cannot see, so the request is rejected rather than silently wrapped around the strip. The
column axis is not concerned, since a strip spans the full width of the window.

In [ ]:
full_shape = fft_array_filter_output_shape(
    (ds_in.height, ds_in.width), kernel, win=None,
    boundary="wrap", out_mode="same",
)
memfile_wrap, ds_out_wrap = open_output(full_shape, ds_in.transform)

try:
    fft_filtering_oa_strip_chain(
        ds_in=ds_in,
        ds_out=ds_out_wrap,
        band=1,
        kernel=kernel,
        boundary="wrap",
        out_mode="same",
        win=None,
        strip_size=64,
        round_out=False,
    )
except NotImplementedError as error:
    print(error)

The same call goes through when the window is processed in one piece, since the core then
reads the whole axis and wraps it correctly.

In [ ]:
fft_filtering_oa_strip_chain(
    ds_in=ds_in,
    ds_out=ds_out_wrap,
    band=1,
    kernel=kernel,
    boundary="wrap",
    out_mode="same",
    win=None,
    strip_size=0,
    round_out=False,
)

wrapped = read_back(memfile_wrap, ds_out_wrap)
wrapped.shape

## Current limitations

The chain adds two restrictions of its own, both met above: the `"valid"` output mode is
not implemented, and a `"wrap"` boundary on the row axis is refused on a striped window.

Strips are also processed one after the other, so a run uses a single core beyond whatever
threading the backend does inside a transform.

The deeper limitations are inherited from the core layer, and the core tutorial *Zooming
out and aliasing* states them. The three of them share a single cause: the SciPy backends
expose one call going from the input image to the filtered image, with no access to the
intermediate spectrum.

Zooming in is unavailable, as it requires padding the spectrum before the inverse
transform. Decimation is applied after a full resolution convolution, so the discarded
samples are computed anyway, and the overlap-add reconstruction of the chain has to run at
full resolution for the same reason. And the decimation phase is limited to an integer
shift, whereas a phase ramp applied to the spectrum would give an exact sub-pixel
centering.

A backend operating directly on the spectrum would lift all three at once. Adding one is
part of the GridR roadmap, without a target version at this stage. The chain would gain
from it twice, since the seams and the discarded samples are both computation it currently
performs and throws away.